In [28]:
#Efetuando a conexão com o banco de dados

import duckdb

con = duckdb.connect("../data/olist.duckdb", read_only=True)

def q(sql: str):
    """Executa uma consulta e devolve um DataFrame para exibir no notebook"""
    return con.sql(sql).df()

In [29]:
# Quais tabelas exitem e quantas colunas e  linhas cada uma tem

q("""
SELECT table_name AS tabela, count(*) AS colunas
FROM information_schema.columns
WHERE table_schema = 'bronze'
GROUP BY table_name
ORDER BY table_name
""")


,tabela,colunas
0,category_translation,4
1,customers,7
2,geolocation,7
3,order_items,9
4,order_payments,7
5,order_reviews,9
6,orders,10
7,products,11
8,sellers,6


In [30]:
q("SELECT * FROM  bronze.orders LIMIT 5")

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,_source_file,_ingested_at
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,olist_orders_dataset.csv,2026-11-19 22:25:52.018471-04:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,olist_orders_dataset.csv,2026-11-19 22:25:52.018471-04:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,olist_orders_dataset.csv,2026-11-19 22:25:52.018471-04:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00,olist_orders_dataset.csv,2026-11-19 22:25:52.018471-04:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00,olist_orders_dataset.csv,2026-11-19 22:25:52.018471-04:00


In [31]:
# Colunas originais do Olist

q("""
SELECT table_name AS tabela, column_name AS coluna
FROM information_schema.columns
WHERE table_schema = 'bronze'
  AND column_name NOT LIKE '\\_%' ESCAPE '\\'
ORDER BY table_name, ordinal_position
""")

,tabela,coluna
0,category_translation,product_category_name
1,category_translation,product_category_name_english
2,customers,customer_id
3,customers,customer_unique_id
4,customers,customer_zip_code_prefix
5,customers,customer_city
6,customers,customer_state
7,geolocation,geolocation_zip_code_prefix
8,geolocation,geolocation_lat
9,geolocation,geolocation_lng


In [32]:
# Verificando se a chave de cada tabela é realmente única, se linhas for > que chaves_unicas, há duplicatas.

q("""
SELECT 'orders' AS tabela, count(*) AS linhas, count(DISTINCT order_id) AS chaves_unicas FROM bronze.orders
UNION ALL SELECT 'customers', count(*), count(DISTINCT customer_id) FROM bronze.customers
UNION ALL SELECT 'products', count(*), count(DISTINCT product_id) FROM bronze.products
UNION ALL SELECT 'sellers', count(*), count(DISTINCT seller_id) FROM bronze.sellers
UNION ALL SELECT 'order_reviews', count(*), count(DISTINCT review_id) FROM bronze.order_reviews
UNION ALL SELECT 'order_items', count(*), count(DISTINCT order_id || '-' || order_item_id) FROM bronze.order_items
""")

,tabela,linhas,chaves_unicas
0,orders,99441,99441
1,customers,99441,99441
2,products,32951,32951
3,sellers,3095,3095
4,order_reviews,99224,98410
5,order_items,112650,112650


In [33]:
# Verificação de nulos em perdidos

q("SUMMARIZE bronze.orders")

,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,order_id,VARCHAR,00010242fe8c5a6d1ba2dd792cb16214,fffe41c64501cc87c801fd61db3f6244,99974,NaN,None,NaN,NaN,NaN,99441,0.00
1,customer_id,VARCHAR,00012a2ce6f8dcda20d059ce98491703,ffffe8b65bbe3087b653a978c870db99,97769,NaN,None,NaN,NaN,NaN,99441,0.00
2,order_status,VARCHAR,approved,unavailable,8,NaN,None,NaN,NaN,NaN,99441,0.00
3,order_purchase_timestamp,VARCHAR,2016-09-04 21:15:19,2018-10-17 17:30:18,99726,NaN,None,NaN,NaN,NaN,99441,0.00
4,order_approved_at,VARCHAR,2016-09-15 12:16:38,2018-09-03 17:40:06,89379,NaN,None,NaN,NaN,NaN,99441,0.16
5,order_delivered_carrier_date,VARCHAR,2016-10-08 10:34:01,2018-09-11 19:48:28,81973,NaN,None,NaN,NaN,NaN,99441,1.79
6,order_delivered_customer_date,VARCHAR,2016-10-11 13:46:32,2018-10-17 13:22:46,91987,NaN,None,NaN,NaN,NaN,99441,2.98
7,order_estimated_delivery_date,VARCHAR,2016-09-30 00:00:00,2018-11-12 00:00:00,460,NaN,None,NaN,NaN,NaN,99441,0.00
8,_source_file,VARCHAR,olist_orders_dataset.csv,olist_orders_dataset.csv,1,NaN,None,NaN,NaN,NaN,99441,0.00
9,_ingested_at,TIMESTAMP WITH TIME ZONE,2026-11-19 22:25:52.018471-04,2026-11-19 22:25:52.018471-04,1,2026-11-19 22:25:52.018471-04,None,2026-11-19 22:25:52.018471-04,2026-11-19 22:25:52.018471-04,2026-11-19 22:25:52.018471-04,99441,0.00


In [34]:
# Verificação de status de pedidos e quantos possuem data de entrega

q("""
SELECT
    order_status,
    count(*) AS pedidos,
    count(*) FILTER (WHERE order_delivered_customer_date IS NULL) AS sem_data_entrega
FROM bronze.orders
GROUP BY order_status
ORDER BY pedidos DESC
""")

,order_status,pedidos,sem_data_entrega
0,delivered,96478,8
1,shipped,1107,1107
2,canceled,625,619
3,unavailable,609,609
4,invoiced,314,314
5,processing,301,301
6,created,5,5
7,approved,2,2


In [35]:
# Verificação de categorias em traduão

q("""
SELECT p.product_category_name, count(*) AS produtos
FROM bronze.products p
LEFT JOIN bronze.category_translation t USING (product_category_name)
WHERE t.product_category_name IS NULL
GROUP BY 1
ORDER BY 2 DESC
""")

,product_category_name,produtos
0,NaN,610
1,portateis_cozinha_e_preparadores_de_alimentos,10
2,pc_gamer,3


In [38]:
# Avaliação por pedido

q("""
SELECT avaliacoes_no_pedido, count(*) AS pedidos
FROM (
    SELECT order_id, count(*) AS avaliacoes_no_pedido
    FROM bronze.order_reviews
    GROUP BY order_id
)
GROUP BY 1
ORDER BY 1
""")

,avaliacoes_no_pedido,pedidos
0,1,98126
1,2,543
2,3,4


In [ ]:
# Verificação de clientes

q("""
SELECT
    count(DISTINCT customer_id) AS customer_id_distintos,
    count(DISTINCT customer_unique_id) AS clientes_reais
FROM bronze.customers
""")

,customer_id_distintos,clientes_reais
0,99441,96096


In [42]:
# Verificação de pagamentos

q("""
SELECT
    payment_type,
    count(*) AS pagamentos,
    count(*) FILTER (WHERE TRY_CAST(payment_value AS DOUBLE) IS NULL) AS valor_invalido,
    count(*) FILTER (WHERE TRY_CAST(payment_value AS DOUBLE) = 0) AS valor_zero
FROM bronze.order_payments
GROUP BY 1
ORDER BY 2 DESC
""")

,payment_type,pagamentos,valor_invalido,valor_zero
0,credit_card,76795,0,0
1,boleto,19784,0,0
2,voucher,5775,0,6
3,debit_card,1529,0,0
4,not_defined,3,0,3


In [45]:
# Verificação de periodo coberto

q("""
SELECT
    strftime(CAST(order_purchase_timestamp AS TIMESTAMP), '%Y-%m') AS mes,
    count(*) AS pedidos
FROM bronze.orders
GROUP BY 1
ORDER BY 1
""")

,mes,pedidos
0,2016-09,4
1,2016-10,324
2,2016-12,1
3,2017-01,800
4,2017-02,1780
5,2017-03,2682
6,2017-04,2404
7,2017-05,3700
8,2017-06,3245
9,2017-07,4026
